# S1-Scout: Reconnaissance for Revenue
### An AI Sales Intelligence Agent for XYZ Analytics Consulting

**Hackathon submission — Build an AI-Powered Sales Intelligence Agent for an Automotive Analytics Consultancy**

S1-Scout automates the top of XYZ Analytics Consulting's B2B sales funnel for the Indian automotive market. It researches the industry, identifies high-potential OEMs, Tier-1 suppliers, and component manufacturers, and maps each company's business challenges to the most relevant consulting solution — **Warranty Analytics**, **Supply-Chain Risk Prediction**, or **Dealer & Field Service Intelligence** — grounded in the official Product & Solutions Handbook via RAG.

---
## Pipeline

| Stage | Agent | Tools | Output |
|---|---|---|---|
| 0. Setup | — | — | Handbook RAG index (ChromaDB) |
| 1. Market Research | Market Research Analyst | web search | Industry report + candidate long-list (~25–30 cos) |
| 2. Company Research | Company Intelligence Researcher | web search | Structured company profiles (JSON) |
| 3. Prioritization | Prospect Prioritization Strategist | — (weighted rubric) | Top 10–15 ranked prospects |
| 4. Solution Mapping | Solution Mapping Consultant | **handbook RAG** | Challenge → solution mappings with handbook citations |
| 5. Reporting | Report Writer | — | All four deliverables (markdown) |

**How to run (Colab):** `Runtime → Run all`. You will be prompted for two API keys (LLM + Serper) unless they are stored in Colab Secrets (🔑 icon) as `GEMINI_API_KEY` / `SERPER_API_KEY`. Upload `XYZ_Analytics_Consulting___Product___Solutions_Handbook.pdf` when prompted (or place it beside the notebook).

**How to run (local Jupyter):** same — keys are read from environment variables or requested via `getpass`; the handbook is read from the notebook directory.

---
# Stage 0 — Environment Setup & Knowledge Layer

Installs dependencies, configures API keys safely (no hardcoded secrets), and builds the RAG index over the Product & Solutions Handbook. This index is exposed to downstream agents as a `handbook_search` tool — the mechanism by which company challenges get mapped to XYZ's service offerings.

In [1]:
# 0.1 — Dependencies (≈2–3 min on a fresh Colab runtime)
%pip install -q "crewai[litellm]>=0.100.0" chromadb pypdf requests pydantic litellm --upgrade

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4

In [2]:
import importlib.util

def is_installed(pkg):
    return importlib.util.find_spec(pkg) is not None

packages = ["crewai", "chromadb", "pypdf", "requests", "pydantic"]

for p in packages:
    print(f"{p}: {'Installed' if is_installed(p) else 'Not installed'}")


crewai: Installed
chromadb: Installed
pypdf: Installed
requests: Installed
pydantic: Installed


In [3]:
from google.colab import drive
drive.mount('/content/drive')
HANDBOOK_FILE = "/content/drive/MyDrive/S1-Scout/XYZ_Analytics_Consulting_Handbook.pdf"

Mounted at /content/drive


In [4]:
# 0.2 — Imports & environment detection
import os, json, re, time, textwrap, pathlib
import requests
from pypdf import PdfReader

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# --- Path configuration ---
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    BASE = pathlib.Path("/content/drive/MyDrive/S1 Scout")
else:
    BASE = pathlib.Path(".")

WORKDIR = BASE / "s1scout_run"
WORKDIR.mkdir(exist_ok=True)

# ← exact filename from Drive
HANDBOOK_FILE = BASE / "XYZ Analytics Consulting_Handbook.pdf"

print(f"Environment : {'Google Colab' if IN_COLAB else 'Local Jupyter'}")
print(f"Base folder : {BASE.resolve()}")
print(f"Work dir    : {WORKDIR.resolve()}")
print(f"Handbook    : {HANDBOOK_FILE} {'✔' if HANDBOOK_FILE.exists() else '✗ NOT FOUND'}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Environment : Google Colab
Base folder : /content/drive/MyDrive/S1 Scout
Work dir    : /content/drive/MyDrive/S1 Scout/s1scout_run
Handbook    : /content/drive/MyDrive/S1 Scout/XYZ Analytics Consulting_Handbook.pdf ✔


In [5]:
# 0.3 — API keys (never hardcoded)
# Resolution order: Colab Secrets -> environment variable -> interactive getpass prompt.
from getpass import getpass

def get_secret(name: str, prompt: str) -> str:
    if IN_COLAB:
        try:
            from google.colab import userdata
            val = userdata.get(name)
            if val:
                return val
        except Exception:
            pass
    return getpass(f"{prompt} ({name}): ")

GROQ_API_KEY   = get_secret("GROQ_API_KEY",  "Enter your Groq API key")
SERPER_API_KEY = get_secret("SERPER_API_KEY", "Enter your Serper API key")

# Push to os.environ so CrewAI/LiteLLM and our tools can read them
os.environ["GROQ_API_KEY"]   = GROQ_API_KEY
os.environ["SERPER_API_KEY"] = SERPER_API_KEY

print("Keys loaded ✔")

Keys loaded ✔


In [6]:
# 0.4 — Run configuration
from crewai import LLM

# Single LLM — Groq Llama 3.3 70B for all stages
llm_fast  = LLM(model="groq/llama-3.3-70b-versatile", temperature=0.2)
llm_smart = LLM(model="groq/llama-3.3-70b-versatile", temperature=0.3)

# Pipeline config
TARGET_TIERS    = ["large", "mid", "small", "private"]
CHUNK_SIZE      = 1100
CHUNK_OVERLAP   = 200
LONGLIST_QUOTAS = {"OEM": 10, "Tier-1 Supplier": 9, "Component Manufacturer": 9}
SHORTLIST_SIZE  = 12
SEARCH_DELAY_S  = 1.0

SERVICES = [
    "Warranty Analytics",
    "Supply-Chain Risk Prediction",
    "Dealer & Field Service Intelligence"
]

print(f"Fast LLM  : {llm_fast.model}")
print(f"Smart LLM : {llm_smart.model}")
print(f"Longlist  : {sum(LONGLIST_QUOTAS.values())} companies ({LONGLIST_QUOTAS})")
print(f"Shortlist : {SHORTLIST_SIZE} companies (final)")

Fast LLM  : groq/llama-3.3-70b-versatile
Smart LLM : groq/llama-3.3-70b-versatile
Longlist  : 28 companies ({'OEM': 10, 'Tier-1 Supplier': 9, 'Component Manufacturer': 9})
Shortlist : 12 companies (final)


## 0.5 — Knowledge Layer: RAG over the Product & Solutions Handbook

The handbook is the ground truth for what XYZ sells. We extract its text, chunk it with overlap (so capability lists, KPI tables, and case studies stay intact), and index it in an in-memory ChromaDB collection. The Solution Mapping agent (Stage 4) queries this index and must cite retrieved passages in its recommendations — this is what keeps challenge→solution reasoning grounded rather than hallucinated.

In [7]:
# 0.5a — Locate & extract the handbook
handbook_path = pathlib.Path(HANDBOOK_FILE)
if not handbook_path.exists():
    if IN_COLAB:
        from google.colab import files
        print("Please upload the Product & Solutions Handbook PDF:")
        uploaded = files.upload()
        handbook_path = pathlib.Path(next(iter(uploaded)))
    else:
        raise FileNotFoundError(f"Place '{HANDBOOK_FILE}' next to this notebook.")

reader = PdfReader(str(handbook_path))
pages = [(i + 1, p.extract_text() or "") for i, p in enumerate(reader.pages)]
raw_chars = sum(len(t) for _, t in pages)
print(f"Handbook loaded: {len(pages)} pages, {raw_chars:,} characters")

Handbook loaded: 20 pages, 37,811 characters


In [8]:
# 0.5b — Clean & chunk (overlapping windows, page-tagged for citation)
BOILERPLATE = re.compile(
    r"(Automotive Analytics Solutions\s+Confidential|© 2025 Analytics Consultancy\. All rights reserved\.\s*Page \d+)"
)

def clean(text: str) -> str:
    text = BOILERPLATE.sub(" ", text)
    return re.sub(r"[ \t]+", " ", text).strip()

chunks, metadatas = [], []
for page_no, text in pages:
    text = clean(text)
    if len(text) < 40:            # skip near-empty pages
        continue
    start = 0
    while start < len(text):
        piece = text[start:start + CHUNK_SIZE]
        chunks.append(piece)
        metadatas.append({"page": page_no})
        if start + CHUNK_SIZE >= len(text):
            break
        start += CHUNK_SIZE - CHUNK_OVERLAP

print(f"{len(chunks)} chunks (size {CHUNK_SIZE}, overlap {CHUNK_OVERLAP})")

49 chunks (size 1100, overlap 200)


In [9]:
# 0.5c — Build the vector index (ChromaDB, in-memory, default MiniLM embeddings)
import chromadb

chroma = chromadb.Client()
try:
    chroma.delete_collection("handbook")   # idempotent re-runs
except Exception:
    pass
handbook_db = chroma.create_collection("handbook", metadata={"hnsw:space": "cosine"})
handbook_db.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    documents=chunks,
    metadatas=metadatas,
)
print(f"Indexed {handbook_db.count()} chunks ✔")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:08<00:00, 10.0MiB/s]


Indexed 49 chunks ✔


In [10]:
# 0.5d — Retrieval function + smoke tests
def handbook_search(query: str, k: int = 4) -> str:
    """Return the k most relevant handbook passages for a query, tagged with page numbers."""
    res = handbook_db.query(query_texts=[query], n_results=k)
    out = []
    for doc, meta in zip(res["documents"][0], res["metadatas"][0]):
        out.append(f"[Handbook p.{meta['page']}] {doc}")
    return "\n\n".join(out)

for q in [
    "How does Warranty Analytics detect defects early and what KPIs are tracked?",
    "supplier financial stress and disruption simulation",
    "improving extended warranty sales across dealer network",
]:
    print("Q:", q)
    print(handbook_search(q, k=2)[:600], "\n" + "-" * 80)

Q: How does Warranty Analytics detect defects early and what KPIs are tracked?
[Handbook p.7] re symptoms to surface hidden defect patterns 
• Correlation analysis linking defects to specific suppliers, plants, or production batches 
• Predictive models estimating failure probability within a defined window (e.g., next 6 months) 
• Early-warning alerts triggered by statistically significant claim spikes 
 
Business Value 
McKinsey research shows that OEMs using AI-driven quality management can reduce warranty expenses by 
5–10% or more. Root-cause analysis that typically takes 7+ weeks manually can be compressed to days. One 
OEM that linked battery telematics to warrant 
--------------------------------------------------------------------------------
Q: supplier financial stress and disruption simulation
[Handbook p.9]  on-time delivery, financial health, and external signals 
Parts Availability Percentage of required parts available on hand at any given time 
Days of Inventory Number

In [11]:
# 0.6 — Agent tools: handbook RAG + Serper web search
from crewai.tools import tool

@tool("handbook_search")
def handbook_search_tool(query: str) -> str:
    """Search XYZ Analytics Consulting's Product & Solutions Handbook. Use this to find service
    capabilities, business value, KPIs, deliverables, case studies and ROI figures that match a
    company's business challenges. Input: a natural-language query."""
    return handbook_search(query, k=4)

@tool("web_search")
def web_search_tool(query: str) -> str:
    """Search the web (Google via Serper). Use for Indian automotive industry research, company
    profiles, financials, manufacturing footprint and recent news. Input: a search query string."""
    time.sleep(SEARCH_DELAY_S)
    r = requests.post(
        "https://google.serper.dev/search",
        headers={"X-API-KEY": os.environ["SERPER_API_KEY"], "Content-Type": "application/json"},
        json={"q": query, "num": 8, "gl": "in"},
        timeout=30,
    )
    r.raise_for_status()
    data = r.json()
    lines = []
    if "answerBox" in data:
        ab = data["answerBox"]
        lines.append(f"ANSWER: {ab.get('answer') or ab.get('snippet','')}")
    for item in data.get("organic", [])[:8]:
        lines.append(f"- {item.get('title','')} | {item.get('snippet','')} ({item.get('link','')})")
    return "\n".join(lines) if lines else "No results."

print(web_search_tool.run("India automotive industry production 2026 SIAM"))

- Press Releases | Monthly Performance: May 2026 Production: The total production of Passenger Vehicles1, Three Wheelers, Two Wheelers, and Quadricycle in May 2026 was 29,27,711 ... (https://www.siam.in/press-release.aspx?mpgid=48&pgidtrail=50)
- Society of Indian Automobile Manufactures | Production. The industry produced a total 21,481,526 vehicles including passenger vehicles, commercial vehicles, three wheelers and two wheelers in ... (https://www.siam.in/statistics.aspx?mpgid=8&pgidtrail=9)
- Society of Indian Automobile Manufactures | Automobile Production Trends ; Passenger Vehicles, 18,38,593, 23,57,411 ; Commercial Vehicles, 4,16,870, 5,67,556 ; Three Wheelers, 4,97,020, 6,19,194 ; Two ... (https://www.siam.in/statistics.aspx?mpgid=8&pgidtrail=13)
- Auto Industry Sees Double-Digit Growth in April 2026 Sales - LinkedIn | In the first month of FY 2026–27, the auto industry recorded robust double-digit domestic sales growth across Passenger Vehicles, ... (https://www.linkedin.com

In [12]:
# 0.7 — LLM handle + JSON cache helpers (structured hand-offs between stages)
from crewai import Agent, Task, Crew, Process, LLM

# LLM handles (defined here so all stages can import from one place)
llm_fast  = LLM(model="groq/llama-3.3-70b-versatile", temperature=0.2)  # Stages 1 & 2
llm_smart = LLM(model="groq/llama-3.3-70b-versatile", temperature=0.3)  # Stages 3, 4 & 5

# JSON cache helpers — structured hand-offs between stages, saved to Drive
def save_json(name: str, obj) -> None:
    (WORKDIR / name).write_text(json.dumps(obj, indent=2, ensure_ascii=False))
    print(f"saved -> {WORKDIR / name}")

def load_json(name: str):
    p = WORKDIR / name
    return json.loads(p.read_text()) if p.exists() else None

def extract_json(text: str):
    """Robustly pull JSON object/array out of LLM response — handles ``` fences."""
    text = re.sub(r"```(?:json)?", "", text).strip("` \n")
    m = re.search(r"[\[{].*[\]}]", text, re.DOTALL)
    return json.loads(m.group()) if m else json.loads(text)

print(f"Fast LLM  : {llm_fast.model}")
print(f"Smart LLM : {llm_smart.model}")
print("Cache helpers ready ✔")

Fast LLM  : groq/llama-3.3-70b-versatile
Smart LLM : groq/llama-3.3-70b-versatile
Cache helpers ready ✔


In [13]:
# 0.7b — PATCH: CrewAI bug #5886 (cache_breakpoint sent to Groq)
# CrewAI 1.14.x adds a 'cache_breakpoint' key to messages that only Anthropic's
# API understands; Groq rejects it with invalid_request_error. We strip it from
# every outgoing litellm call. Remove this cell once the bug is fixed upstream.
import litellm

if not getattr(litellm, "_s1scout_patched", False):
    _orig_completion = litellm.completion

    def _clean_completion(*args, **kwargs):
        for m in kwargs.get("messages") or []:
            if isinstance(m, dict):
                m.pop("cache_breakpoint", None)
        return _orig_completion(*args, **kwargs)

    litellm.completion = _clean_completion
    litellm._s1scout_patched = True

print("litellm patched — cache_breakpoint stripped for Groq ✔")

litellm patched — cache_breakpoint stripped for Groq ✔


---
# Stage 1 — Market Research Agent
**Agent:** Market Research Analyst · **Tools:** `web_search` · **Outputs:** `industry_report.md`, `candidate_longlist.json`

Researches the Indian automotive industry (size, growth, EV transition, policy, pressure points) and produces (a) the Market Research Report deliverable and (b) a long-list of ~25–30 candidate companies across the three target segments — OEMs, Tier-1 suppliers, component manufacturers — each tagged with the pain signals that make them worth investigating.

*Implementation lands in Phase 3.*

In [14]:
# TODO(Phase 3): Market Research Analyst — agent definition, tasks, run, cache to JSON
# market_report_md, candidate_longlist = ...
# save_json("candidate_longlist.json", candidate_longlist)

In [15]:
# 1.0 — Pre-fetch industry data (7 searches, cached — never re-fetches)
INDUSTRY_SOURCES = {
    "siam_production" : "India automotive production statistics 2026 SIAM",
    "acma_components" : "ACMA Indian auto component industry overview 2026",
    "ibef_overview"   : "IBEF India automobile industry market size 2026",
    "ev_growth"       : "India EV electric vehicle sales growth 2026",
    "warranty_issues" : "Indian OEM warranty recall quality issues 2025 2026",
    "supply_chain"    : "India auto supply chain disruption logistics cost 2026",
    "dealer_network"  : "India automotive dealer network challenges expansion 2026",
}

industry_data = load_json("industry_raw_data.json")
if industry_data:
    print(f"Cache hit — industry data already fetched ({len(industry_data)} topics)")
else:
    industry_data = {}
    for key, query in INDUSTRY_SOURCES.items():
        print(f"  Fetching: {query[:55]}...")
        industry_data[key] = web_search_tool.run(query)
        time.sleep(SEARCH_DELAY_S)
    save_json("industry_raw_data.json", industry_data)
    print(f"Saved industry_raw_data.json ✔  ({len(industry_data)} topics)")

Cache hit — industry data already fetched (7 topics)


In [16]:
# 1.1 — Company Universe (embedded knowledge asset)
# Curated one-time from: IBEF "Major players by segment" & manufacturing-cluster
# maps (ACMA-sourced), SIAM OEM membership, Moneycontrol sector classification
# (210+ listed stocks / 34 sub-industries), NSE/BSE listings. Compiled 2026-07.
# Format: (name, segment, sub_vertical, cluster, listed, cap_tier)
#   cap_tier ~ mid-2026 mcap: large >₹50k cr | mid ₹10–50k | small <₹10k | private
_U = [
 # ── OEMs (SIAM members) ─────────────────────────────────────────
 ("Maruti Suzuki India","OEM","Passenger Vehicles","North",True,"large"),
 ("Tata Motors","OEM","PV + Commercial Vehicles","West",True,"large"),
 ("Mahindra & Mahindra","OEM","PV + CV + Farm","West",True,"large"),
 ("Hyundai Motor India","OEM","Passenger Vehicles","South",True,"large"),
 ("Hero MotoCorp","OEM","Two-Wheelers","North",True,"large"),
 ("Bajaj Auto","OEM","Two & Three-Wheelers","West",True,"large"),
 ("TVS Motor Company","OEM","Two-Wheelers","South",True,"large"),
 ("Ashok Leyland","OEM","Commercial Vehicles","South",True,"large"),
 ("Eicher Motors","OEM","2W Premium + CV","North",True,"large"),
 ("Force Motors","OEM","LCV + Van","West",True,"small"),
 ("Escorts Kubota","OEM","Tractors + Construction","North",True,"mid"),
 ("SML Isuzu","OEM","Commercial Vehicles","North",True,"small"),
 ("Ola Electric","OEM","Electric Two-Wheelers","South",True,"mid"),
 ("Piaggio Vehicles","OEM","Three-Wheelers + LCV","West",False,"private"),
 # ── Tier-1 Suppliers ────────────────────────────────────────────
 ("Bosch Limited","Tier-1 Supplier","Powertrain + Electricals","South",True,"large"),
 ("Samvardhana Motherson","Tier-1 Supplier","Wiring Harness + Modules","North",True,"large"),
 ("Bharat Forge","Tier-1 Supplier","Forgings + Chassis","West",True,"large"),
 ("Uno Minda","Tier-1 Supplier","Switches + Lighting + Electronics","North",True,"mid"),
 ("Sona BLW (Sona Comstar)","Tier-1 Supplier","Driveline + EV Motors","North",True,"mid"),
 ("Varroc Engineering","Tier-1 Supplier","Electrical + Polymer","West",True,"mid"),
 ("Endurance Technologies","Tier-1 Supplier","Suspension + Braking + Castings","West",True,"mid"),
 ("Schaeffler India","Tier-1 Supplier","Bearings + Powertrain","West",True,"mid"),
 ("SKF India","Tier-1 Supplier","Bearings","West",True,"mid"),
 ("ZF CV Control Systems India","Tier-1 Supplier","CV Braking Systems","South",True,"mid"),
 ("Lucas TVS","Tier-1 Supplier","Electricals + Starters","South",False,"private"),
 ("Exide Industries","Tier-1 Supplier","Batteries","East",True,"mid"),
 ("Amara Raja Energy & Mobility","Tier-1 Supplier","Batteries","South",True,"mid"),
 ("Tata AutoComp Systems","Tier-1 Supplier","Multi-product Systems","West",False,"private"),
 # ── Component Manufacturers (IBEF segment map) ──────────────────
 ("Sundram Fasteners","Component Manufacturer","Fasteners + Powertrain Parts","South",True,"mid"),
 ("Rane Holdings","Component Manufacturer","Steering + Brake Lining + Valves","South",True,"small"),
 ("Gabriel India","Component Manufacturer","Shock Absorbers","North",True,"small"),
 ("Jamna Auto Industries","Component Manufacturer","Leaf Springs","North",True,"small"),
 ("Lumax Industries","Component Manufacturer","Headlights + Lighting","North",True,"small"),
 ("Pricol Limited","Component Manufacturer","Sensors + Instrument Clusters","South",True,"small"),
 ("Suprajit Engineering","Component Manufacturer","Cables + Actuation","South",True,"mid"),
 ("Subros","Component Manufacturer","AC / Thermal Systems","North",True,"small"),
 ("Ramkrishna Forgings","Component Manufacturer","Forgings","East",True,"mid"),
 ("Steel Strips Wheels","Component Manufacturer","Wheels","North",True,"small"),
 ("Minda Corporation","Component Manufacturer","Locks + Electronics + Wiring","North",True,"small"),
 ("Automotive Axles","Component Manufacturer","Axles (CV)","South",True,"small"),
 ("GNA Axles","Component Manufacturer","Axle Shafts","North",True,"small"),
 ("Shriram Pistons & Rings","Component Manufacturer","Pistons + Engine Parts","North",True,"small"),
 ("Sandhar Technologies","Component Manufacturer","Locking + Sheet Metal","North",True,"small"),
 ("Fiem Industries","Component Manufacturer","2W Lighting","North",True,"small"),
 ("JBM Auto","Component Manufacturer","Sheet Metal + Buses","North",True,"mid"),
 ("Wheels India","Component Manufacturer","Wheels + Air Suspension","South",True,"small"),
 ("Sundaram Clayton","Component Manufacturer","Die Castings","South",True,"mid"),
 ("Craftsman Automation","Component Manufacturer","Machining + Powertrain","South",True,"mid"),
 ("Happy Forgings","Component Manufacturer","Forgings (heavy)","North",True,"small"),
 ("Talbros Automotive","Component Manufacturer","Gaskets + Chassis","North",True,"small"),
 ("Apollo Tyres","Component Manufacturer","Tyres","North",True,"mid"),
 ("CEAT Limited","Component Manufacturer","Tyres","West",True,"mid"),
]

KEYS = ["name","segment","sub_vertical","cluster","listed","cap_tier"]
companies_all = [dict(zip(KEYS, row)) for row in _U]
save_json("company_universe.json", companies_all)   # persisted as run artifact

from collections import Counter
print(f"Universe: {len(companies_all)} companies")
print(f"Segments : {dict(Counter(c['segment'] for c in companies_all))}")
print(f"Tiers    : {dict(Counter(c['cap_tier'] for c in companies_all))}")

saved -> /content/drive/MyDrive/S1 Scout/s1scout_run/company_universe.json
Universe: 52 companies
Segments : {'OEM': 14, 'Tier-1 Supplier': 14, 'Component Manufacturer': 24}
Tiers    : {'large': 12, 'small': 18, 'mid': 19, 'private': 3}


In [17]:
# 1.2 — Screening: universe → ICP filter → balanced long-list (28)
# Deterministic, explainable rules: tier filter (ICP), segment quotas,
# prefer listed (financials retrievable), maximize sub-vertical diversity.
LONGLIST_QUOTAS = {"OEM": 10, "Tier-1 Supplier": 9, "Component Manufacturer": 9}

def screen_universe(companies, quotas):
    longlist = []
    for segment, quota in quotas.items():
        pool = [c for c in companies if c["segment"] == segment]
        pool.sort(key=lambda c: (not c["listed"]))       # listed first
        picked, seen_subs = [], set()
        for c in pool:                                    # pass 1: new sub-verticals
            sub = c["sub_vertical"].split(" + ")[0]
            if sub not in seen_subs and len(picked) < quota:
                picked.append(c); seen_subs.add(sub)
        for c in pool:                                    # pass 2: fill remainder
            if c not in picked and len(picked) < quota:
                picked.append(c)
        longlist += picked
    return longlist

# ICP filter first (TARGET_TIERS from config cell 0.4), then quota screening
pool_all = [c for c in companies_all if c["cap_tier"] in TARGET_TIERS]
longlist_companies = screen_universe(pool_all, LONGLIST_QUOTAS)

save_json("longlist_input.json", longlist_companies)
print(f"ICP pool : {len(pool_all)} of {len(companies_all)} (tiers: {TARGET_TIERS})")
print(f"Long-list: {len(longlist_companies)} companies")
for seg in LONGLIST_QUOTAS:
    names = [c["name"] for c in longlist_companies if c["segment"] == seg]
    print(f"  {seg} ({len(names)}): {', '.join(names)}")

saved -> /content/drive/MyDrive/S1 Scout/s1scout_run/longlist_input.json
ICP pool : 52 of 52 (tiers: ['large', 'mid', 'small', 'private'])
Long-list: 28 companies
  OEM (10): Maruti Suzuki India, Tata Motors, Hero MotoCorp, Bajaj Auto, Ashok Leyland, Eicher Motors, Force Motors, Escorts Kubota, Ola Electric, Piaggio Vehicles
  Tier-1 Supplier (9): Bosch Limited, Samvardhana Motherson, Bharat Forge, Uno Minda, Sona BLW (Sona Comstar), Varroc Engineering, Endurance Technologies, Schaeffler India, ZF CV Control Systems India
  Component Manufacturer (9): Sundram Fasteners, Rane Holdings, Gabriel India, Jamna Auto Industries, Lumax Industries, Pricol Limited, Suprajit Engineering, Subros, Ramkrishna Forgings


In [18]:
# 1.3 — Agent definition + Task 1: Industry Report
from crewai import Agent, Task, Crew, Process

market_analyst = Agent(
    role="Senior Automotive Market Research Analyst",
    goal="Synthesize Indian automotive industry research into a market report and evidence-backed company long-list.",
    backstory=(
        "You are a seasoned analyst covering the Indian automotive sector. "
        "You identify business pain points — warranty costs, supply disruptions, "
        "dealer underperformance — and match them to analytics opportunities. "
        "You only state facts you can trace to evidence, never generic claims."
    ),
    tools=[web_search_tool],
    llm=llm_fast,
    verbose=True,
    max_iter=5,
)

industry_context = json.dumps(industry_data, indent=2)
task_industry_report = Task(
    description=(
        "Using the pre-fetched industry research below, write a structured "
        "Market Research Report in markdown.\n\n"
        f"INDUSTRY DATA:\n{industry_context}\n\n"
        "Cover these 5 sections:\n"
        "1. Executive Summary (key numbers, 3-4 sentences)\n"
        "2. Market Overview — size, production volume, growth, segments\n"
        "3. Key Market Pressures — exactly 5: warranty costs | supply chain | "
        "EV transition | dealer performance | demand volatility\n"
        "4. Analytics Opportunities — why data solutions are needed now\n"
        "5. Target Customer Segments\n\n"
        "Use web_search_tool only to verify a specific number. "
        "Minimum 500 words. Cite figures from the data provided."
    ),
    expected_output=(
        "Complete markdown Market Research Report, all 5 sections, "
        "minimum 500 words, specific figures, no hallucinated statistics."
    ),
    agent=market_analyst,
)
print("Agent + Task 1 defined ✔")

Agent + Task 1 defined ✔


In [19]:
# 1.4 — Task 2: evidence-grounded pain-signal research
# Groq quality test showed the LLM fabricates plausible pain points when
# unsupervised — every claim must come from search results, with URL.
company_list_str = json.dumps(
    [{"name": c["name"], "segment": c["segment"], "sub_vertical": c["sub_vertical"]}
     for c in longlist_companies], indent=2)

task_longlist = Task(
    description=(
        "Research each company below for SPECIFIC, EVIDENCED business pain points.\n\n"
        f"COMPANY LIST:\n{company_list_str}\n\n"
        "For EACH company, search: '<name> <sub_vertical> recall warranty supply chain news 2025 2026'\n\n"
        "STRICT EVIDENCE RULES:\n"
        "- Report ONLY pain signals found in your search results, with the source URL\n"
        "- Each pain signal must reference a concrete event, figure, or announcement\n"
        "- If nothing specific is found, set pain_signals to ['no specific signal found'] "
        "— DO NOT invent plausible-sounding challenges\n"
        "- Use company names EXACTLY as given in the list\n"
        "- A company may fit one, two, or all three services — do not force variety\n\n"
        f"XYZ services: {SERVICES}\n\n"
        "Return a raw JSON array only — no markdown fences, no explanation.\n"
        "Keys per object: name | segment | sub_vertical | pain_signals (list) | "
        "potential_services (list) | source_urls (list)"
    ),
    expected_output=(
        f"Raw JSON array of {len(longlist_companies)} objects. Every pain signal "
        "traceable to a search result. No markdown fences."
    ),
    agent=market_analyst,
    context=[task_industry_report],
)
print("Task 2 defined ✔")

Task 2 defined ✔


In [20]:
# 1.5 — Run Task 1 only: Industry Report (cache-aware)
if (WORKDIR / "industry_report.md").exists():
    industry_report_md = (WORKDIR / "industry_report.md").read_text()
    print(f"Cache hit — report loaded ({len(industry_report_md):,} chars)")
else:
    crew_report = Crew(agents=[market_analyst], tasks=[task_industry_report],
                       process=Process.sequential, verbose=True)
    await crew_report.kickoff_async()
    industry_report_md = task_industry_report.output.raw
    (WORKDIR / "industry_report.md").write_text(industry_report_md)
    print("industry_report.md saved ✔")

Cache hit — report loaded (6,369 chars)


In [21]:
# 1.6 — Pre-fetch per-company evidence searches (cached incrementally)
company_search_raw = load_json("company_search_raw.json") or {}

for c in longlist_companies:
    if c["name"] in company_search_raw:
        continue
    q = f"{c['name']} {c['sub_vertical']} recall warranty supply chain news 2025 2026"
    print(f"  Fetching: {c['name']}")
    company_search_raw[c["name"]] = web_search_tool.run(q)
    save_json("company_search_raw.json", company_search_raw)   # save after EACH — resumable
    time.sleep(SEARCH_DELAY_S)

print(f"Evidence cached for {len(company_search_raw)} companies ✔")

Evidence cached for 28 companies ✔


In [22]:
# 1.7 — Batched pain-signal analysis (5 companies per LLM call, stays under Groq's 12k TPM)
BATCH_SIZE = 5
candidate_longlist = load_json("candidate_longlist.json") or []
done_names = {c["name"] for c in candidate_longlist}
todo = [c for c in longlist_companies if c["name"] not in done_names]

def analyze_batch(batch):
    evidence = "\n\n".join(
        f"### {c['name']} ({c['segment']} | {c['sub_vertical']})\n{company_search_raw[c['name']]}"
        for c in batch)
    prompt = (
        "You are an automotive market analyst. Below are web search results for "
        f"{len(batch)} Indian automotive companies.\n\n{evidence}\n\n"
        "For EACH company, extract SPECIFIC pain signals — concrete events, figures, "
        "or announcements found in the results (recalls, production halts, supply "
        "disruptions, warranty events, showroom closures, cost pressures, losses).\n"
        "RULES:\n"
        "- Only report what appears in the search results, with its source URL\n"
        "- If nothing specific: pain_signals = ['no specific signal found']\n"
        "- Use company names EXACTLY as given\n"
        f"- Map to applicable services from {SERVICES} — do not force variety\n\n"
        "Return ONLY a raw JSON array (no fences). Keys per object: "
        "name, segment, sub_vertical, pain_signals, potential_services, source_urls"
    )
    return extract_json(llm_fast.call([{"role": "user", "content": prompt}]))

for i in range(0, len(todo), BATCH_SIZE):
    batch = todo[i:i + BATCH_SIZE]
    print(f"Batch {i//BATCH_SIZE + 1}: {', '.join(c['name'] for c in batch)}")
    try:
        candidate_longlist.extend(analyze_batch(batch))
        save_json("candidate_longlist.json", candidate_longlist)   # incremental — resumable
    except Exception as e:
        print(f"  Batch failed ({e}) — waiting 60s, retrying once...")
        time.sleep(60)
        candidate_longlist.extend(analyze_batch(batch))
        save_json("candidate_longlist.json", candidate_longlist)
    time.sleep(25)   # ~2 batches/min keeps us under 12k TPM

print(f"\nLong-list complete: {len(candidate_longlist)} companies ✔")


Long-list complete: 28 companies ✔


In [23]:
# 1.8 — Preview Stage 1 outputs
from IPython.display import Markdown, display

print("=" * 60)
print("MARKET RESEARCH REPORT (first 1500 chars)")
print("=" * 60)
display(Markdown(industry_report_md[:1500] + "\n\n*...full report in s1scout_run/industry_report.md*"))

print("\n" + "=" * 60)
print(f"CANDIDATE LONG-LIST — {len(candidate_longlist)} companies")
print("=" * 60)
for co in candidate_longlist[:5]:
    print(f"  {co.get('name')} [{co.get('segment')}]")
    signals = co.get('pain_signals', [])
    print(f"    Pain : {signals[0] if signals else 'N/A'}")
    print(f"    Fit  : {', '.join(co.get('potential_services', []))}")
    print()
if len(candidate_longlist) > 5:
    print(f"  ... and {len(candidate_longlist)-5} more in candidate_longlist.json")

MARKET RESEARCH REPORT (first 1500 chars)


# Executive Summary
The Indian automotive industry has witnessed significant growth in recent years, with a total production of 21,481,526 vehicles in 2025, including passenger vehicles, commercial vehicles, three wheelers, and two wheelers. The industry has seen a double-digit growth in April 2026 sales, with Maruti Suzuki India Limited and Tata Motors being the biggest share gainers. The total production of Passenger Vehicles, Three Wheelers, Two Wheelers, and Quadricycle in May 2026 was 29,27,711 units. The Indian automotive industry is expected to reach US$ 300 billion by 2026, with the auto components industry expected to grow 7-9% in FY26.

# Market Overview
The Indian automotive industry is one of the largest in the world, with a significant production volume and growth rate. The industry produced a total of 21,481,526 vehicles in 2025, including passenger vehicles, commercial vehicles, three wheelers, and two wheelers. The production of passenger vehicles, commercial vehicles, three wheelers, and two wheelers has been increasing over the years, with a growth rate of 6.8% in H1 FY26. The industry is dominated by domestic players, with Maruti Suzuki India Limited and Tata Motors being the leading manufacturers. The Indian automotive industry is also a significant exporter, with exports of auto components growing by 9.3% to USD 12.1 billion in H1 FY26.

The industry can be segmented into several categories, including passenger vehicles, commercial vehicles, three wheeler

*...full report in s1scout_run/industry_report.md*


CANDIDATE LONG-LIST — 28 companies
  Maruti Suzuki India [OEM]
    Pain : recall of 39,506 units of the Grand Vitara
    Fit  : Warranty Analytics, Supply-Chain Risk Prediction

  Tata Motors [OEM]
    Pain : cyberattack at Jaguar Land Rover
    Fit  : Supply-Chain Risk Prediction, Dealer & Field Service Intelligence

  Hero MotoCorp [OEM]
    Pain : consumer wins against Hero MotoCorp
    Fit  : Warranty Analytics

  Bajaj Auto [OEM]
    Pain : severe shortage of rare earth magnets
    Fit  : Supply-Chain Risk Prediction

  Ashok Leyland [OEM]
    Pain : no specific signal found
    Fit  : no specific service found

  ... and 23 more in candidate_longlist.json


---
# Stage 2 — Company Research Agent
**Agent:** Company Intelligence Researcher · **Tools:** `web_search` · **Output:** `company_profiles.json`

Loops over the long-list. For each company, gathers: overview, products & services, manufacturing locations, growth initiatives, financial highlights, recent news, and inferred business challenges. Results are cached per company so an interrupted run resumes instead of restarting.

*Implementation lands in Phase 4.*

In [24]:
# TODO(Phase 4): per-company research loop with incremental caching
# company_profiles = ...
# save_json("company_profiles.json", company_profiles)

In [28]:
# 2.0 — Pre-fetch company research data (3 queries per company, cached incrementally)
company_research_raw = load_json("company_research_raw.json") or {}

for c in longlist_companies:
    name = c["name"]
    if name in company_research_raw:
        continue   # cache hit — skip

    print(f"  Fetching: {name}...")
    company_research_raw[name] = {
        "profile": web_search_tool.run(
            f"{name} company overview products manufacturing plants locations India automotive"),
        "financials": web_search_tool.run(
            f"{name} revenue profit market cap FY2026 annual results screener moneycontrol"),
        "news": web_search_tool.run(
            f"{name} India expansion growth investment challenges news 2025 2026"),
    }
    save_json("company_research_raw.json", company_research_raw)
    time.sleep(SEARCH_DELAY_S)

print(f"Research data cached for {len(company_research_raw)} companies ✔")

  Fetching: Maruti Suzuki India...
saved -> /content/drive/MyDrive/S1 Scout/s1scout_run/company_research_raw.json
  Fetching: Tata Motors...
saved -> /content/drive/MyDrive/S1 Scout/s1scout_run/company_research_raw.json
  Fetching: Hero MotoCorp...
saved -> /content/drive/MyDrive/S1 Scout/s1scout_run/company_research_raw.json
  Fetching: Bajaj Auto...
saved -> /content/drive/MyDrive/S1 Scout/s1scout_run/company_research_raw.json
  Fetching: Ashok Leyland...
saved -> /content/drive/MyDrive/S1 Scout/s1scout_run/company_research_raw.json
  Fetching: Eicher Motors...
saved -> /content/drive/MyDrive/S1 Scout/s1scout_run/company_research_raw.json
  Fetching: Force Motors...
saved -> /content/drive/MyDrive/S1 Scout/s1scout_run/company_research_raw.json
  Fetching: Escorts Kubota...
saved -> /content/drive/MyDrive/S1 Scout/s1scout_run/company_research_raw.json
  Fetching: Ola Electric...
saved -> /content/drive/MyDrive/S1 Scout/s1scout_run/company_research_raw.json
  Fetching: Piaggio Vehicles

In [29]:
# 2.1 — Company Research Agent
from crewai import Agent

company_researcher = Agent(
    role="Automotive Company Intelligence Researcher",
    goal=(
        "Extract structured, evidence-based company profiles from search results. "
        "Every field must be grounded in the provided search data — never invent facts."
    ),
    backstory=(
        "You are a B2B research analyst specializing in Indian automotive companies. "
        "You extract precise, factual profiles from web search snippets — "
        "manufacturing locations, revenue figures, recent investments, and business challenges. "
        "You never guess or fill gaps with generic statements."
    ),
    tools=[],         # no tools — works from pre-fetched evidence only
    llm=llm_fast,
    verbose=False,
    max_iter=3,
)
print("Company Research Agent defined ✔")

Company Research Agent defined ✔


In [31]:
# 2.2 — Batched profile extraction (3 companies per LLM call)
BATCH_SIZE = 3

company_profiles = load_json("company_profiles.json") or []
done_names = {p["name"] for p in company_profiles}
todo = [c for c in longlist_companies if c["name"] not in done_names]

if not todo:
    print(f"Cache hit — {len(company_profiles)} company profiles already extracted")
else:
    def extract_profiles_batch(batch):
        evidence_blocks = []
        for c in batch:
            raw = company_research_raw.get(c["name"], {})
            block = (
                f"### {c['name']} | {c['segment']} | {c['sub_vertical']}\n"
                f"PROFILE SEARCH:\n{raw.get('profile','no data')}\n\n"
                f"FINANCIALS SEARCH:\n{raw.get('financials','no data')}\n\n"
                f"NEWS SEARCH:\n{raw.get('news','no data')}"
            )
            evidence_blocks.append(block)

        evidence = "\n\n" + "="*60 + "\n\n".join(evidence_blocks)

        prompt = (
            f"Extract structured profiles for these {len(batch)} Indian automotive companies "
            f"from the search results below.\n\n{evidence}\n\n"
            "For EACH company return a JSON object with EXACTLY these keys:\n"
            "- name: string (exact as given)\n"
            "- segment: string\n"
            "- sub_vertical: string\n"
            "- overview: 2-3 sentence company description\n"
            "- products_services: list of 3-5 main products/services\n"
            "- manufacturing_locations: list of plant cities/states\n"
            "- business_growth: 1-2 sentences on recent expansion or investments\n"
            "- financial_highlights: revenue, profit, market cap with FY year. "
            "Write 'not available in search results' if not found\n"
            "- recent_news: list of 2-3 specific news items with year\n"
            "- business_challenges: list of 2-3 specific challenges from evidence. "
            "Write ['no specific challenge found'] if none\n\n"
            "RULES:\n"
            "- Only use facts from the search results\n"
            "- Never invent locations, figures, or news\n"
            "- Use company names EXACTLY as given\n\n"
            "Return ONLY a raw JSON array — no markdown fences, no explanation."
        )
        return extract_json(llm_fast.call([{"role": "user", "content": prompt}]))

    print(f"Extracting profiles for {len(todo)} companies in batches of {BATCH_SIZE}...")
    for i in range(0, len(todo), BATCH_SIZE):
        batch = todo[i:i + BATCH_SIZE]
        print(f"  Batch {i//BATCH_SIZE + 1}: {', '.join(c['name'] for c in batch)}")
        try:
            results = extract_profiles_batch(batch)
            company_profiles.extend(results)
            save_json("company_profiles.json", company_profiles)
        except Exception as e:
            print(f"  Batch failed ({e}) — waiting 90s, retrying...")
            time.sleep(90)
            try:
                results = extract_profiles_batch(batch)
                company_profiles.extend(results)
                save_json("company_profiles.json", company_profiles)
            except Exception as e2:
                print(f"  Retry failed ({e2}) — skipping, continuing...")
        time.sleep(45)

    print(f"\ncompany_profiles.json saved ✔  ({len(company_profiles)} profiles)")

Extracting profiles for 18 companies in batches of 3...
  Batch 1: Bosch Limited, Samvardhana Motherson, Bharat Forge
saved -> /content/drive/MyDrive/S1 Scout/s1scout_run/company_profiles.json
  Batch 2: Uno Minda, Sona BLW (Sona Comstar), Varroc Engineering
saved -> /content/drive/MyDrive/S1 Scout/s1scout_run/company_profiles.json
  Batch 3: Endurance Technologies, Schaeffler India, ZF CV Control Systems India
saved -> /content/drive/MyDrive/S1 Scout/s1scout_run/company_profiles.json
  Batch 4: Sundram Fasteners, Rane Holdings, Gabriel India
saved -> /content/drive/MyDrive/S1 Scout/s1scout_run/company_profiles.json
  Batch 5: Jamna Auto Industries, Lumax Industries, Pricol Limited
saved -> /content/drive/MyDrive/S1 Scout/s1scout_run/company_profiles.json
  Batch 6: Suprajit Engineering, Subros, Ramkrishna Forgings
saved -> /content/drive/MyDrive/S1 Scout/s1scout_run/company_profiles.json

company_profiles.json saved ✔  (28 profiles)


In [32]:
# 2.3 — Preview Stage 2 outputs
print(f"COMPANY PROFILES — {len(company_profiles)} companies")
print("=" * 60)
for p in company_profiles[:3]:
    print(f"\n## {p.get('name')} [{p.get('segment')}]")
    print(f"   Overview   : {p.get('overview','')[:120]}...")
    print(f"   Products   : {', '.join(p.get('products_services',[])[:3])}")
    locs = p.get('manufacturing_locations', [])
    print(f"   Plants     : {', '.join(locs) if locs else 'not found'}")
    print(f"   Financials : {str(p.get('financial_highlights',''))[:100]}")
    challenges = p.get('business_challenges', [])
    print(f"   Challenge  : {challenges[0] if challenges else 'none'}")
print(f"\n  ... and {max(0,len(company_profiles)-3)} more in company_profiles.json")

COMPANY PROFILES — 28 companies

## Maruti Suzuki India [OEM]
   Overview   : Maruti Suzuki India is the country's leading passenger vehicle manufacturer with manufacturing facilities in Gurgaon and...
   Products   : Alto, WagonR, Eeco
   Plants     : Gurgaon, Manesar, Gujarat
   Financials : Revenue: ₹1,83316 Cr, Profit: ₹14680 Cr, Market Cap: ₹4,57162 Crore (FY26)
   Challenge  : Margin pressure

## Tata Motors [OEM]
   Overview   : Tata Motors is India's leading manufacturer of commercial vehicles and has a significant presence in the passenger vehic...
   Products   : Trucks, Buses, Passenger Vehicles
   Plants     : Jamshedpur, Lucknow, Pantnagar, Dharwad, Pune
   Financials : Revenue: ₹1,05,447 crore, Profit: ₹8,470 crore, Market Cap: ₹1,56444 Crore (FY26)
   Challenge  : Competition in the commercial vehicle market

## Hero MotoCorp [OEM]
   Overview   : Hero MotoCorp is the world's largest manufacturer of two-wheelers and has a significant presence in India. The company h...
 

---
# Stage 3 — Prioritization Agent
**Agent:** Prospect Prioritization Strategist · **Tools:** none (transparent reasoning over Stage 2 evidence) · **Output:** `top_prospects.json`

Scores every profiled company against an explicit weighted rubric and returns the Top 10–15:

| Criterion | Weight | Signal |
|---|---|---|
| Pain-point evidence | 30% | candidate_longlist.json pain_signals |
| Scale & budget capacity | 25% | company_profiles.json financials |
| Solution fit breadth | 25% | How many of 3 services apply |
| Data maturity | 15% | How many of 3 services apply |
| Accessibility | 5% | Digital/EV/DMS mentions in profile |

*Implementation lands in Phase 4.*

In [25]:
# TODO(Phase 4): scoring task with rubric embedded in the prompt; per-criterion scores in output
# top_prospects = ...
# save_json("top_prospects.json", top_prospects)

In [34]:
# 3.0 — Load Stage 1 + Stage 2 outputs and merge into one enriched list
profiles_by_name = {p["name"]: p for p in company_profiles}
longlist_by_name = {c["name"]: c for c in candidate_longlist}

enriched = []
for name, profile in profiles_by_name.items():
    pain_data = longlist_by_name.get(name, {})
    enriched.append({
        **profile,
        "pain_signals"      : pain_data.get("pain_signals", ["no signal found"]),
        "potential_services": pain_data.get("potential_services", []),
        "source_urls"       : pain_data.get("source_urls", []),
        "cap_tier"          : next(
            (c["cap_tier"] for c in companies_all if c["name"] == name), "unknown")
    })

by_segment = {}
for c in enriched:
    by_segment.setdefault(c["segment"], []).append(c)

print(f"Enriched: {len(enriched)} companies")
for seg, cos in by_segment.items():
    print(f"  {seg}: {len(cos)} companies")

Enriched: 28 companies
  OEM: 10 companies
  Tier-1 Supplier: 9 companies
  Component Manufacturer: 9 companies


In [35]:
# 3.1 — Prospect Prioritization (one LLM call per segment, no Serper)
SHORTLIST_QUOTAS = {"OEM": 4, "Tier-1 Supplier": 4, "Component Manufacturer": 4}

def score_segment(segment, companies):
    company_summaries = []
    for c in companies:
        summary = (
            f"Company: {c['name']}\n"
            f"Sub-vertical: {c['sub_vertical']}\n"
            f"Cap tier: {c.get('cap_tier','unknown')}\n"
            f"Overview: {c.get('overview','')}\n"
            f"Products: {', '.join(c.get('products_services',[]))}\n"
            f"Plants: {', '.join(c.get('manufacturing_locations',[]))}\n"
            f"Financials: {c.get('financial_highlights','')}\n"
            f"Growth: {c.get('business_growth','')}\n"
            f"Pain signals: {'; '.join(c.get('pain_signals',[]))}\n"
            f"Potential services: {', '.join(c.get('potential_services',[]))}\n"
            f"Recent news: {'; '.join(c.get('recent_news',[]))}\n"
            f"Business challenges: {'; '.join(c.get('business_challenges',[]))}"
        )
        company_summaries.append(summary)

    quota = SHORTLIST_QUOTAS[segment]
    prompt = (
        f"You are a B2B sales strategist for XYZ Analytics Consulting. "
        f"Score and rank these {len(companies)} Indian {segment} companies "
        f"as prospects for analytics consulting services.\n\n"
        f"COMPANIES:\n\n" + "\n\n---\n\n".join(company_summaries) + "\n\n"
        f"SCORING RUBRIC (score each 1-10, weights shown):\n"
        f"1. pain_evidence (30%): Specific, evidenced business pain — recalls, "
        f"supply disruptions, warranty events, production halts. "
        f"Generic challenges score low. 'no signal found' scores 1.\n"
        f"2. scale_budget (25%): Revenue scale and financial capacity to invest "
        f"in analytics. Higher revenue = higher score.\n"
        f"3. solution_fit (25%): How many of the 3 XYZ services apply: "
        f"Warranty Analytics, Supply-Chain Risk Prediction, "
        f"Dealer & Field Service Intelligence. All 3 = 10, two = 7, one = 4.\n"
        f"4. winnability (15%): Mid and small cap companies score higher — "
        f"large caps have in-house data science teams and incumbent vendors. "
        f"mid=8, small=9, large=4, private=6.\n"
        f"5. data_maturity (5%): Evidence of digital initiatives, EV programs, "
        f"DMS/telematics/ERP investments. More digital = higher score.\n\n"
        f"Select the TOP {quota} companies from this segment.\n\n"
        f"Return ONLY a raw JSON array of {quota} objects. "
        f"Each object must have EXACTLY these keys:\n"
        f"- name: string (exact company name)\n"
        f"- segment: string\n"
        f"- rank: integer (1 = best in segment)\n"
        f"- scores: object with keys pain_evidence, scale_budget, solution_fit, "
        f"winnability, data_maturity (each an integer 1-10)\n"
        f"- weighted_score: float (computed: "
        f"pain*0.30 + scale*0.25 + fit*0.25 + win*0.15 + data*0.05)\n"
        f"- why_selected: 2-3 sentences explaining why this company is a strong "
        f"prospect, citing specific evidence\n"
        f"- primary_service: the single most relevant XYZ service\n"
        f"- expected_value: one sentence on expected business value for the client\n\n"
        f"No markdown, no explanation. Raw JSON array only."
    )
    return extract_json(llm_smart.call([{"role": "user", "content": prompt}]))

top_prospects = load_json("top_prospects.json") or []
done_segments = {p["segment"] for p in top_prospects}

for segment in SHORTLIST_QUOTAS:
    if segment in done_segments:
        print(f"Cache hit — {segment} already scored")
        continue
    print(f"Scoring {segment} ({len(by_segment.get(segment,[]))} companies)...")
    try:
        results = score_segment(segment, by_segment[segment])
        top_prospects.extend(results)
        save_json("top_prospects.json", top_prospects)
        print(f"  → {len(results)} prospects selected ✔")
    except Exception as e:
        print(f"  Segment failed ({e}) — waiting 60s, retrying...")
        import time; time.sleep(60)
        results = score_segment(segment, by_segment[segment])
        top_prospects.extend(results)
        save_json("top_prospects.json", top_prospects)
    time.sleep(30)

top_prospects.sort(key=lambda x: x.get("weighted_score", 0), reverse=True)
save_json("top_prospects.json", top_prospects)
print(f"\ntop_prospects.json saved ✔  ({len(top_prospects)} companies shortlisted)")

Scoring OEM (10 companies)...
saved -> /content/drive/MyDrive/S1 Scout/s1scout_run/top_prospects.json
  → 4 prospects selected ✔
Scoring Tier-1 Supplier (9 companies)...
saved -> /content/drive/MyDrive/S1 Scout/s1scout_run/top_prospects.json
  → 4 prospects selected ✔
Scoring Component Manufacturer (9 companies)...
saved -> /content/drive/MyDrive/S1 Scout/s1scout_run/top_prospects.json
  → 4 prospects selected ✔
saved -> /content/drive/MyDrive/S1 Scout/s1scout_run/top_prospects.json

top_prospects.json saved ✔  (12 companies shortlisted)


In [36]:
# 3.2 — Preview Top 12 shortlist
from IPython.display import Markdown, display

lines = ["# S1-Scout: Top 12 Prospect Companies\n"]
lines.append(f"*Scored against 5-criterion rubric · Segment-balanced (4 OEM / 4 Tier-1 / 4 Component)*\n")

for seg in SHORTLIST_QUOTAS:
    seg_prospects = sorted(
        [p for p in top_prospects if p["segment"] == seg],
        key=lambda x: x.get("rank", 99)
    )
    lines.append(f"\n## {seg}\n")
    for p in seg_prospects:
        scores = p.get("scores", {})
        ws = p.get("weighted_score", 0)
        lines.append(
            f"**{p['rank']}. {p['name']}** (score: {ws:.1f}/10)  \n"
            f"*Primary service:* {p.get('primary_service','')}  \n"
            f"*Why selected:* {p.get('why_selected','')}  \n"
            f"*Expected value:* {p.get('expected_value','')}  \n"
            f"Scores — Pain: {scores.get('pain_evidence','?')}/10 · "
            f"Scale: {scores.get('scale_budget','?')}/10 · "
            f"Fit: {scores.get('solution_fit','?')}/10 · "
            f"Winnability: {scores.get('winnability','?')}/10  \n"
        )

display(Markdown("\n".join(lines)))

# S1-Scout: Top 12 Prospect Companies

*Scored against 5-criterion rubric · Segment-balanced (4 OEM / 4 Tier-1 / 4 Component)*


## OEM


## Tier-1 Supplier

**1. Bosch Limited** (score: 7.3/10)  
*Primary service:* Supply-Chain Risk Prediction  
*Why selected:* Bosch Limited is a strong prospect due to its large revenue scale and financial capacity to invest in analytics, as well as its specific business pain signals such as supply chain disruptions and production halts. The company's strong presence in India and its commitment to digital initiatives also make it an attractive client. Additionally, its recent investment in a new manufacturing unit and plans to augment capacity in Jaipur demonstrate its growth potential.  
*Expected value:* By implementing Supply-Chain Risk Prediction, Bosch Limited can expect to reduce its supply chain disruptions and production halts, resulting in significant cost savings and improved efficiency.  
Scores — Pain: 8/10 · Scale: 9/10 · Fit: 4/10 · Winnability: 4/10  

**2. Samvardhana Motherson** (score: 7.8/10)  
*Primary service:* Supply-Chain Risk Prediction  
*Why selected:* Samvardhana Motherson is a strong prospect due to its large revenue scale and financial capacity to invest in analytics, as well as its multiple business pain signals such as demerging its domestic wiring harness business. The company's strong presence in India and its commitment to digital initiatives also make it an attractive client. Additionally, its recent investment in a new automotive lighting plant and plans to invest in capital expenditure demonstrate its growth potential.  
*Expected value:* By implementing Supply-Chain Risk Prediction and Dealer & Field Service Intelligence, Samvardhana Motherson can expect to improve its supply chain efficiency and reduce its logistics costs, resulting in significant cost savings and improved customer satisfaction.  
Scores — Pain: 6/10 · Scale: 10/10 · Fit: 7/10 · Winnability: 4/10  

**3. Uno Minda** (score: 7.0/10)  
*Primary service:* Warranty Analytics  
*Why selected:* Uno Minda is a strong prospect due to its mid-cap size and relatively lower winnability score, making it more accessible to XYZ Analytics Consulting. The company's commitment to digital initiatives and its recent investment in a new seating plant and joint venture also demonstrate its growth potential. Additionally, its multiple business pain signals such as maintaining margin momentum and expansion and growth make it an attractive client.  
*Expected value:* By implementing Warranty Analytics, Uno Minda can expect to reduce its warranty claims and improve its product quality, resulting in significant cost savings and improved customer satisfaction.  
Scores — Pain: 4/10 · Scale: 8/10 · Fit: 7/10 · Winnability: 8/10  

**4. Sona BLW (Sona Comstar)** (score: 6.8/10)  
*Primary service:* Supply-Chain Risk Prediction  
*Why selected:* Sona BLW (Sona Comstar) is a strong prospect due to its mid-cap size and relatively lower winnability score, making it more accessible to XYZ Analytics Consulting. The company's commitment to digital initiatives and its recent investment in new plants and expansion of its capacity demonstrate its growth potential. Additionally, its multiple business pain signals such as tariff uncertainty and geopolitical developments make it an attractive client.  
*Expected value:* By implementing Supply-Chain Risk Prediction and Dealer & Field Service Intelligence, Sona BLW (Sona Comstar) can expect to improve its supply chain efficiency and reduce its logistics costs, resulting in significant cost savings and improved customer satisfaction.  
Scores — Pain: 5/10 · Scale: 7/10 · Fit: 7/10 · Winnability: 8/10  


## Component Manufacturer


In [37]:
# Run this once after 3.2 — fixes segment names to canonical values
SEGMENT_MAP = {
    "Indian Component Manufacturer": "Component Manufacturer",
    "Passenger Vehicles": "OEM",
    "Two & Three-Wheelers": "OEM",
    "Electric Two-Wheelers": "OEM",
    "PV + Commercial Vehicles": "OEM",
}
for p in top_prospects:
    p["segment"] = SEGMENT_MAP.get(p["segment"], p["segment"])
save_json("top_prospects.json", top_prospects)
print("Segments normalized ✔")
print({p["name"]: p["segment"] for p in top_prospects})

saved -> /content/drive/MyDrive/S1 Scout/s1scout_run/top_prospects.json
Segments normalized ✔
{'Ramkrishna Forgings': 'Component Manufacturer', 'Tata Motors': 'OEM', 'Samvardhana Motherson': 'Tier-1 Supplier', 'Pricol Limited': 'Component Manufacturer', 'Maruti Suzuki India': 'OEM', 'Bosch Limited': 'Tier-1 Supplier', 'Rane Holdings': 'Component Manufacturer', 'Ola Electric': 'OEM', 'Uno Minda': 'Tier-1 Supplier', 'Bajaj Auto': 'OEM', 'Sona BLW (Sona Comstar)': 'Tier-1 Supplier', 'Sundram Fasteners': 'Component Manufacturer'}


---
# Stage 4 — Solution Mapping Agent  ⭐ core of "Use of Product Knowledge"
**Agent:** Solution Mapping Consultant · **Tools:** `handbook_search` (RAG) · **Output:** `solution_mappings.json`

For each shortlisted company: takes its identified challenges, queries the handbook index, and selects the most relevant consulting solution. Every recommendation must cite the handbook — capability, KPI, or case study — that addresses the specific challenge (e.g. *"correlation analysis linking defects to specific suppliers, plants, or production batches" [p.7] ↔ Company X's supplier-quality recall*), plus an expected-business-value statement anchored to handbook ROI figures.

*Implementation lands in Phase 5.*

In [26]:
# TODO(Phase 5): mapping loop — challenges -> handbook_search -> recommendation + citations
# solution_mappings = ...
# save_json("solution_mappings.json", solution_mappings)

In [39]:
# 4.0 — Solution Mapping Agent (handbook RAG — the core of "Use of Product Knowledge")
solution_mappings = load_json("solution_mappings.json") or []
done_names = {m["name"] for m in solution_mappings}
todo_prospects = [p for p in top_prospects if p["name"] not in done_names]

if not todo_prospects:
    print(f"Cache hit — {len(solution_mappings)} solution mappings already complete")
else:
    print(f"Mapping solutions for {len(todo_prospects)} companies...")
    for prospect in todo_prospects:
        name = prospect["name"]
        print(f"  Mapping: {name}...")

        # Step 1: retrieve relevant handbook passages
        pain_query = " ".join(prospect.get("pain_signals", []))[:300]
        service_query = prospect.get("primary_service", "analytics consulting")
        handbook_context = (
            handbook_search(f"{service_query} capabilities KPIs business value", k=3)
            + "\n\n"
            + handbook_search(f"{pain_query} analytics solution", k=2)
        )

        # Step 2: get full profile for context
        profile = profiles_by_name.get(name, {})
        pain_signals = prospect.get("pain_signals", [])
        scores = prospect.get("scores", {})

        prompt = (
            f"You are a solution consultant at XYZ Analytics Consulting.\n\n"
            f"COMPANY: {name}\n"
            f"Segment: {prospect['segment']} | Sub-vertical: {profile.get('sub_vertical','')}\n"
            f"Overview: {profile.get('overview','')}\n"
            f"Products: {', '.join(profile.get('products_services',[]))}\n"
            f"Plants: {', '.join(profile.get('manufacturing_locations',[]))}\n"
            f"Financials: {profile.get('financial_highlights','')}\n"
            f"Pain signals: {'; '.join(pain_signals)}\n"
            f"Business challenges: {'; '.join(profile.get('business_challenges',[]))}\n\n"
            f"HANDBOOK KNOWLEDGE (XYZ's service offerings):\n{handbook_context}\n\n"
            f"PRIMARY SERVICE RECOMMENDED: {prospect.get('primary_service','')}\n\n"
            f"Write a solution mapping recommendation. Return a JSON object with "
            f"EXACTLY these keys:\n"
            f"- name: company name\n"
            f"- segment: company segment\n"
            f"- recommended_service: the primary XYZ service\n"
            f"- executive_summary: 3-4 sentences — company context + why XYZ + "
            f"expected outcome\n"
            f"- business_challenges: list of 2-3 specific challenges this company faces\n"
            f"- solution_mapping: list of 2-3 objects, each with:\n"
            f"  - challenge: the specific company challenge\n"
            f"  - xyz_capability: the exact XYZ capability that addresses it "
            f"(cite the handbook passage)\n"
            f"  - handbook_citation: the handbook page/section where this is described\n"
            f"  - kpi_impact: which KPI(s) will improve\n"
            f"- expected_business_value: specific ROI statement anchored to handbook "
            f"figures (e.g. '5-10% warranty cost reduction' or '15% logistics savings')\n"
            f"- why_winnable: one sentence on why {name} would engage XYZ over "
            f"a large incumbent\n\n"
            f"RULES:\n"
            f"- Every xyz_capability must be quoted or closely paraphrased from "
            f"the handbook passages provided\n"
            f"- Every expected_business_value must cite a handbook ROI figure\n"
            f"- Be specific to {name}'s actual situation — no generic consulting language\n\n"
            f"Return ONLY raw JSON — no markdown fences."
        )

        try:
            result = extract_json(
                llm_smart.call([{"role": "user", "content": prompt}])
            )
            solution_mappings.append(result)
            save_json("solution_mappings.json", solution_mappings)
        except Exception as e:
            print(f"    Failed ({e}) — waiting 45s, retrying...")
            time.sleep(45)
            result = extract_json(
                llm_smart.call([{"role": "user", "content": prompt}])
            )
            solution_mappings.append(result)
            save_json("solution_mappings.json", solution_mappings)

        time.sleep(20)   # ~3 calls/min — safe under Groq 12k TPM

    print(f"\nsolution_mappings.json saved ✔  ({len(solution_mappings)} mappings)")

Mapping solutions for 12 companies...
  Mapping: Ramkrishna Forgings...
saved -> /content/drive/MyDrive/S1 Scout/s1scout_run/solution_mappings.json
  Mapping: Tata Motors...
saved -> /content/drive/MyDrive/S1 Scout/s1scout_run/solution_mappings.json
  Mapping: Samvardhana Motherson...
saved -> /content/drive/MyDrive/S1 Scout/s1scout_run/solution_mappings.json
  Mapping: Pricol Limited...
saved -> /content/drive/MyDrive/S1 Scout/s1scout_run/solution_mappings.json
  Mapping: Maruti Suzuki India...
saved -> /content/drive/MyDrive/S1 Scout/s1scout_run/solution_mappings.json
  Mapping: Bosch Limited...
saved -> /content/drive/MyDrive/S1 Scout/s1scout_run/solution_mappings.json
  Mapping: Rane Holdings...
saved -> /content/drive/MyDrive/S1 Scout/s1scout_run/solution_mappings.json
  Mapping: Ola Electric...
    Failed (Expecting ',' delimiter: line 30 column 4 (char 1726)) — waiting 45s, retrying...
saved -> /content/drive/MyDrive/S1 Scout/s1scout_run/solution_mappings.json
  Mapping: Uno Min

In [40]:
# 4.1 — Preview solution mappings
from IPython.display import Markdown, display

if solution_mappings:
    m = solution_mappings[0]   # show first company as sample
    lines = [
        f"# Solution Mapping Sample: {m['name']}\n",
        f"**Segment:** {m['segment']}  ",
        f"**Recommended Service:** {m['recommended_service']}\n",
        f"## Executive Summary\n{m.get('executive_summary','')}\n",
        f"## Business Challenges\n" +
        "\n".join(f"- {c}" for c in m.get('business_challenges',[])) + "\n",
        f"## Solution Mapping\n"
    ]
    for sm in m.get("solution_mapping", []):
        lines.append(f"**Challenge:** {sm.get('challenge','')}")
        lines.append(f"**XYZ Capability:** {sm.get('xyz_capability','')}")
        lines.append(f"**Handbook Citation:** {sm.get('handbook_citation','')}")
        lines.append(f"**KPI Impact:** {sm.get('kpi_impact','')}\n")
    lines.append(
        f"## Expected Business Value\n{m.get('expected_business_value','')}\n")
    lines.append(
        f"## Why Winnable\n{m.get('why_winnable','')}\n")
    lines.append(f"---\n*{len(solution_mappings)} of 12 mappings complete*")

    display(Markdown("\n".join(lines)))

# Solution Mapping Sample: Ramkrishna Forgings

**Segment:** Component Manufacturer | Sub-vertical: Forgings  
**Recommended Service:** Supply-Chain Risk Prediction

## Executive Summary
Ramkrishna Forgings, a leading manufacturer of forged, machined, and fabricated components, can benefit from XYZ's Supply-Chain Risk Prediction service to mitigate potential disruptions in their supply chain. With 22 manufacturing facilities across India, the company is vulnerable to supplier failures, material shortages, and logistics delays. By leveraging XYZ's expertise, Ramkrishna Forgings can anticipate and respond to potential risks, ensuring production continuity and reducing excess inventory. This will enable the company to maintain its strong global presence and competitiveness in the automotive and farm equipment industries.

## Business Challenges
- Supplier failures and material shortages
- Logistics delays and disruptions
- Excess inventory and production continuity

## Solution Mapping

**Challenge:** Supplier failures and material shortages
**XYZ Capability:** Real-time risk scoring for suppliers, components, and logistics routes
**Handbook Citation:** Handbook p.9
**KPI Impact:** Supplier Risk Score, Parts Availability

**Challenge:** Logistics delays and disruptions
**XYZ Capability:** Disruption simulation — modelling the production impact of a supplier failure or material shortage
**Handbook Citation:** Handbook p.9
**KPI Impact:** Days of Inventory

**Challenge:** Excess inventory and production continuity
**XYZ Capability:** Predictive lead-time forecasting for critical parts
**Handbook Citation:** Handbook p.9
**KPI Impact:** Parts Availability, Days of Inventory

## Expected Business Value
By implementing XYZ's Supply-Chain Risk Prediction service, Ramkrishna Forgings can expect to cut logistics costs by up to 15% and reduce excess inventory by approximately 35%, as seen in industry analysis (Handbook p.9)

## Why Winnable
Ramkrishna Forgings would engage XYZ over a large incumbent because of XYZ's specialized expertise in the automotive sector, flexible engagement models, and commitment to data security and intellectual property ownership, as outlined in the handbook (Handbook p.5).

---
*12 of 12 mappings complete*

---
# Stage 5 — Report Generator
**Agent:** Report Writer · **Tools:** none · **Output:** final deliverables rendered below

Assembles the four deliverables into clean markdown, displayed as notebook output so reviewers can evaluate without re-running:
1. **Market Research Report**
2. **Top 10–15 Target Companies** (overview · why selected · recommended solution · justification)
3. **Business Recommendations** per company (executive summary · challenges · solution · expected value)
4. Files saved to `s1scout_run/` for the GitHub repo

*Implementation lands in Phase 5.*

In [27]:
# TODO(Phase 5): assemble + display final deliverables
# from IPython.display import Markdown, display
# display(Markdown(final_report_md))

In [41]:
# 5.0 — Report Generator (pure assembly — zero API calls)
from IPython.display import Markdown, display
from datetime import date

# ── helpers ──────────────────────────────────────────────
def clean_segment(s):
    """Normalize segment strings that crept in from LLM output."""
    if "Component" in s: return "Component Manufacturer"
    if "Tier-1" in s or "Tier 1" in s: return "Tier-1 Supplier"
    if "OEM" in s: return "OEM"
    return s

def get_profile(name):
    return profiles_by_name.get(name, {})

# ── Deliverable 1: Market Research Report ────────────────
report_md = industry_report_md   # already saved from Stage 1

# ── Deliverable 2 & 3: Top Companies + Recommendations ──
company_sections = []
solution_by_name = {m["name"]: m for m in solution_mappings}

for rank_overall, prospect in enumerate(top_prospects, 1):
    name = prospect["name"]
    seg  = clean_segment(prospect.get("segment",""))
    sol  = solution_by_name.get(name, {})
    prof = get_profile(name)
    scores = prospect.get("scores", {})
    ws = prospect.get("weighted_score", 0)

    section = f"""
---
## {rank_overall}. {name}
**Segment:** {seg} | **Sub-vertical:** {prof.get('sub_vertical','')}
**Recommended Service:** {sol.get('recommended_service', prospect.get('primary_service',''))}
**Prospect Score:** {ws:.1f}/10

### Company Overview
{prof.get('overview','')}

**Products & Services:** {', '.join(prof.get('products_services',[]))}
**Manufacturing Locations:** {', '.join(prof.get('manufacturing_locations',[])) or 'See company website'}
**Financial Highlights:** {prof.get('financial_highlights','')}
**Business Growth:** {prof.get('business_growth','')}

### Why Selected
{sol.get('executive_summary', prospect.get('why_selected',''))}

**Scoring Breakdown:**
| Criterion | Score | Weight |
|---|---|---|
| Pain-point Evidence | {scores.get('pain_evidence','?')}/10 | 30% |
| Scale & Budget | {scores.get('scale_budget','?')}/10 | 25% |
| Solution Fit | {scores.get('solution_fit','?')}/10 | 25% |
| Winnability | {scores.get('winnability','?')}/10 | 15% |
| Data Maturity | {scores.get('data_maturity','?')}/10 | 5% |

### Business Challenges
{chr(10).join(f'- {c}' for c in sol.get('business_challenges', prof.get('business_challenges',[])))}

### Solution Mapping
"""
    for sm in sol.get("solution_mapping", []):
        section += f"""
**Challenge:** {sm.get('challenge','')}
- *XYZ Capability:* {sm.get('xyz_capability','')}
- *Handbook Reference:* {sm.get('handbook_citation','')}
- *KPI Impact:* {sm.get('kpi_impact','')}
"""
    section += f"""
### Expected Business Value
{sol.get('expected_business_value', prospect.get('expected_value',''))}

### Why XYZ Will Win This Account
{sol.get('why_winnable', '')}

**Recent News:** {'; '.join(prof.get('recent_news',[])[:2])}
"""
    company_sections.append(section)

# ── Assemble full report ──────────────────────────────────
today = date.today().strftime("%B %d, %Y")
full_report = f"""# S1-Scout: Reconnaissance for Revenue
## AI Sales Intelligence Report — XYZ Analytics Consulting
*Generated: {today} | Indian Automotive Market*

---

# Part 1: Market Research Report

{report_md}

---

# Part 2: Top 12 Target Companies

*Shortlisted from a universe of 52 companies across 3 segments and 34 sub-verticals.
Screened using a 5-criterion weighted rubric with segment quotas (4 OEM / 4 Tier-1 / 4 Component).*

{"".join(company_sections)}

---

# Part 3: Business Recommendations Summary

| Rank | Company | Segment | Recommended Service | Score |
|---|---|---|---|---|
"""

for i, p in enumerate(top_prospects, 1):
    seg = clean_segment(p.get("segment",""))
    sol = solution_by_name.get(p["name"],{})
    svc = sol.get("recommended_service", p.get("primary_service",""))
    full_report += f"| {i} | {p['name']} | {seg} | {svc} | {p.get('weighted_score',0):.1f}/10 |\n"

full_report += f"""

---

# Part 4: Architecture & Methodology

## Agent Pipeline
S1-Scout uses a 5-stage sequential CrewAI pipeline:

1. **Stage 0 — Knowledge Layer:** Handbook PDF → 49 chunks → ChromaDB RAG index
2. **Stage 1 — Market Research:** Pre-fetched SIAM/IBEF/ACMA industry data →
   Groq synthesis → Industry Report + 28-company evidence-grounded long-list
3. **Stage 2 — Company Research:** 3 Serper queries per company →
   Groq batch extraction → Structured profiles (overview, products, plants, financials, news)
4. **Stage 3 — Prioritization:** 5-criterion weighted rubric, segment-balanced scoring →
   Top 12 shortlist (4 OEM / 4 Tier-1 / 4 Component)
5. **Stage 4 — Solution Mapping:** Pain signals + ChromaDB handbook retrieval →
   Groq generates cited recommendations per company

## Company Universe
Screened from: 210+ listed stocks (34 Moneycontrol sub-industries),
1,000+ ACMA members, SIAM OEM membership.
Final universe: 52 companies across 3 segments, tagged by cap tier and sub-vertical.

## Assumptions & Limitations
- Web data is point-in-time; financial figures are approximate from public snippets
- Large-cap companies (Bosch, Maruti) have low winnability scores by design —
  they have in-house DS teams; XYZ's best accounts are mid/small-cap
- Ashok Leyland showed no specific pain signal in search — honest gap,
  not fabricated (anti-hallucination rule enforced throughout)
- Serper free tier used (~200 of 2,500 queries consumed)
- Groq free tier used (llama-3.3-70b-versatile, 12k TPM on-demand tier)
- ChromaDB in-memory index rebuilt on every Colab runtime (~30 sec)
"""

# Save to Drive
output_path = WORKDIR / "S1_Scout_Final_Report.md"
output_path.write_text(full_report)
print(f"Final report saved ✔  ({len(full_report):,} chars)")
print(f"Path: {output_path}")

Final report saved ✔  (48,758 chars)
Path: /content/drive/MyDrive/S1 Scout/s1scout_run/S1_Scout_Final_Report.md


In [42]:
# 5.1 — Display final deliverables (judges see this without re-running)
print("=" * 70)
print("S1-SCOUT: RECONNAISSANCE FOR REVENUE")
print("AI Sales Intelligence Report — XYZ Analytics Consulting")
print("=" * 70)

# Summary table first
display(Markdown("""
## Top 12 Target Companies — Executive Summary

| Rank | Company | Segment | Recommended Service | Score |
|---|---|---|---|---|
""" + "\n".join(
    f"| {i} | **{p['name']}** | {clean_segment(p.get('segment',''))} | "
    f"{solution_by_name.get(p['name'],{}).get('recommended_service', p.get('primary_service',''))} | "
    f"{p.get('weighted_score',0):.1f}/10 |"
    for i, p in enumerate(top_prospects, 1)
)))

# Full report
display(Markdown(full_report[:8000] + "\n\n*...full report in S1_Scout_Final_Report.md*"))
print(f"\nFull report: {WORKDIR}/S1_Scout_Final_Report.md")

S1-SCOUT: RECONNAISSANCE FOR REVENUE
AI Sales Intelligence Report — XYZ Analytics Consulting



## Top 12 Target Companies — Executive Summary

| Rank | Company | Segment | Recommended Service | Score |
|---|---|---|---|---|
| 1 | **Ramkrishna Forgings** | Component Manufacturer | Supply-Chain Risk Prediction | 8.0/10 |
| 2 | **Tata Motors** | OEM | Supply-Chain Risk Prediction | 7.8/10 |
| 3 | **Samvardhana Motherson** | Tier-1 Supplier | Supply-Chain Risk Prediction | 7.8/10 |
| 4 | **Pricol Limited** | Component Manufacturer | Supply-Chain Risk Prediction | 7.8/10 |
| 5 | **Maruti Suzuki India** | OEM | Warranty Analytics | 7.3/10 |
| 6 | **Bosch Limited** | Tier-1 Supplier | Supply-Chain Risk Prediction | 7.3/10 |
| 7 | **Rane Holdings** | Component Manufacturer | Warranty Analytics | 7.3/10 |
| 8 | **Ola Electric** | OEM | Warranty Analytics | 7.2/10 |
| 9 | **Uno Minda** | Tier-1 Supplier | Warranty Analytics | 7.0/10 |
| 10 | **Bajaj Auto** | OEM | Supply-Chain Risk Prediction | 6.9/10 |
| 11 | **Sona BLW (Sona Comstar)** | Tier-1 Supplier | Supply-Chain Risk Prediction | 6.8/10 |
| 12 | **Sundram Fasteners** | Component Manufacturer | Supply-Chain Risk Prediction | 6.2/10 |

# S1-Scout: Reconnaissance for Revenue
## AI Sales Intelligence Report — XYZ Analytics Consulting
*Generated: July 08, 2026 | Indian Automotive Market*

---

# Part 1: Market Research Report

# Executive Summary
The Indian automotive industry has witnessed significant growth in recent years, with a total production of 21,481,526 vehicles in 2025, including passenger vehicles, commercial vehicles, three wheelers, and two wheelers. The industry has seen a double-digit growth in April 2026 sales, with Maruti Suzuki India Limited and Tata Motors being the biggest share gainers. The total production of Passenger Vehicles, Three Wheelers, Two Wheelers, and Quadricycle in May 2026 was 29,27,711 units. The Indian automotive industry is expected to reach US$ 300 billion by 2026, with the auto components industry expected to grow 7-9% in FY26.

# Market Overview
The Indian automotive industry is one of the largest in the world, with a significant production volume and growth rate. The industry produced a total of 21,481,526 vehicles in 2025, including passenger vehicles, commercial vehicles, three wheelers, and two wheelers. The production of passenger vehicles, commercial vehicles, three wheelers, and two wheelers has been increasing over the years, with a growth rate of 6.8% in H1 FY26. The industry is dominated by domestic players, with Maruti Suzuki India Limited and Tata Motors being the leading manufacturers. The Indian automotive industry is also a significant exporter, with exports of auto components growing by 9.3% to USD 12.1 billion in H1 FY26.

The industry can be segmented into several categories, including passenger vehicles, commercial vehicles, three wheelers, and two wheelers. The passenger vehicle segment is the largest, accounting for a significant share of the total production. The commercial vehicle segment is also significant, with a growing demand for trucks and buses. The three wheeler and two wheeler segments are also important, with a large number of manufacturers operating in these segments.

The Indian automotive industry is also witnessing a significant growth in the electric vehicle (EV) segment, with EV sales reaching 2.64 lakh units in May 2026. The EV segment is expected to grow significantly in the coming years, with the government providing incentives and subsidies to promote the adoption of EVs.

# Key Market Pressures
The Indian automotive industry is facing several key market pressures, including:

* **Warranty Costs**: The industry is facing increasing warranty costs, with a significant number of vehicles being recalled due to quality issues. In 2025, vehicle recalls in India hit an 8-year low, with just 119,173 units recalled across all manufacturers.
* **Supply Chain**: The industry is facing supply chain disruptions, with the West Asia crisis disrupting trade and supply chains across sectors in India. The automotive sector is facing intensified supply chain disruptions, longer shipping timelines, and rising logistics costs.
* **EV Transition**: The industry is facing a significant transition to electric vehicles, with the government promoting the adoption of EVs through incentives and subsidies. The EV segment is expected to grow significantly in the coming years, with EV sales reaching 2.64 lakh units in May 2026.
* **Dealer Performance**: The industry is facing challenges in terms of dealer performance, with dealers facing financing, inventory, and margin challenges. The top challenges faced by automobile dealers in India include financing, inventory, margins, and market challenges.
* **Demand Volatility**: The industry is facing demand volatility, with the demand for vehicles fluctuating significantly over the years. The industry is expected to grow significantly in the coming years, with the Indian automotive industry expected to reach US$ 300 billion by 2026.

# Analytics Opportunities
The Indian automotive industry is facing several analytics opportunities, with data solutions being needed to address the key market pressures. The industry can leverage data analytics to optimize supply chains, predict demand, and improve dealer performance. The industry can also use data analytics to optimize warranty costs, with data analytics helping to identify quality issues and reduce warranty costs.

The industry can also leverage data analytics to promote the adoption of EVs, with data analytics helping to identify areas where EVs can be promoted and incentivized. The industry can also use data analytics to optimize production, with data analytics helping to predict demand and optimize production accordingly.

The use of data analytics in the Indian automotive industry is expected to grow significantly in the coming years, with the industry expected to leverage data analytics to address the key market pressures and promote growth.

# Target Customer Segments
The target customer segments for the Indian automotive industry include:

* **Passenger Vehicle Buyers**: The passenger vehicle segment is the largest in the Indian automotive industry, with a significant number of buyers in this segment. The segment is expected to grow significantly in the coming years, with the demand for passenger vehicles increasing significantly.
* **Commercial Vehicle Buyers**: The commercial vehicle segment is also significant in the Indian automotive industry, with a large number of buyers in this segment. The segment is expected to grow significantly in the coming years, with the demand for commercial vehicles increasing significantly.
* **Electric Vehicle Buyers**: The EV segment is expected to grow significantly in the coming years, with the government promoting the adoption of EVs through incentives and subsidies. The segment is expected to attract a significant number of buyers, with the demand for EVs increasing significantly.
* **Dealers and Distributors**: The dealers and distributors are also an important customer segment for the Indian automotive industry, with the industry relying on them to sell and distribute vehicles. The segment is expected to face significant challenges in the coming years, with the industry expected to leverage data analytics to optimize dealer performance and promote growth.
* **Fleet Owners and Operators**: The fleet owners and operators are also an important customer segment for the Indian automotive industry, with the industry relying on them to purchase and operate vehicles. The segment is expected to grow significantly in the coming years, with the demand for vehicles increasing significantly.

---

# Part 2: Top 12 Target Companies

*Shortlisted from a universe of 52 companies across 3 segments and 34 sub-verticals.
Screened using a 5-criterion weighted rubric with segment quotas (4 OEM / 4 Tier-1 / 4 Component).*


---
## 1. Ramkrishna Forgings
**Segment:** Component Manufacturer | **Sub-vertical:** Forgings
**Recommended Service:** Supply-Chain Risk Prediction
**Prospect Score:** 8.0/10

### Company Overview
Ramkrishna Forgings is a leading manufacturer of forged, machined, and fabricated components, catering to various industries including automotive, farm equipment, and more. The company has a strong global presence, with 22 manufacturing facilities across India. Ramkrishna Forgings is India's biggest integrated forging, casting, and fabrication facility.

**Products & Services:** forged components, machined components, fabricated components, automotive parts, forged wheels
**Manufacturing Locations:** Jamshedpur, Liluah, Chennai
**Financial Highlights:** Revenue: Rs 1077.85 crore, Profit: Rs 0.37 crore, Market Cap: Rs 10377 Crore (FY 2026)
**Business Growth:** Ramkrishna Forgings is establishing Asia's second-largest manufacturing plant in India, with a total project cost of Rs 2,000 crore. The company is expecting 15-20% volume growth in H2, driven by new capacity.

### Why Selected
Ramkrishna Forgings, a leading manufacturer of forged, machined, and fabricated components, can benefit from XYZ's S

*...full report in S1_Scout_Final_Report.md*


Full report: /content/drive/MyDrive/S1 Scout/s1scout_run/S1_Scout_Final_Report.md


---
## Assumptions & Known Limitations
- Web data is point-in-time and search-quality dependent; financial figures are approximate values from public sources.
- Serper free tier (2,500 queries) is sufficient for a full run (~150–250 queries); intermediate JSON caching keeps re-runs cheap.
- The handbook is treated as the sole source of truth for XYZ's offerings; no service capabilities are invented beyond it.